In [ ]:

from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from typing import List, Any
import numpy as np
import chromadb
import os

C:\Users\pavit\AppData\Local\Temp\ipykernel_10932\1842040700.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
d:\AI\YTRag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def Process_all_pdfs(pdf_files):
    path_dir = Path(pdf_files)
    pdf_documents = []
    all_pdf_files = list(path_dir.glob("**/*.pdf"))
    print(path_dir)
    print(f"Found {len(all_pdf_files)} PDF files in {pdf_files}")
    for pdf_file in all_pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()
            print("type of documents:", type(documents))
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            pdf_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    return pdf_documents

pdf_documents_list = Process_all_pdfs("../data")
pdf_documents_list

..\data
Found 2 PDF files in ../data
Processing ..\data\pdf_files\OS 10_5Marks with ans.pdf
type of documents: <class 'list'>
Loaded 27 documents from ..\data\pdf_files\OS 10_5Marks with ans.pdf
Processing ..\data\pdf_files\OS_PCAEA_Repeated_Questions_2021_2024.pdf
type of documents: <class 'list'>
Loaded 2 documents from ..\data\pdf_files\OS_PCAEA_Repeated_Questions_2021_2024.pdf


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-23T18:41:10+00:00', 'source': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf', 'file_path': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf', 'total_pages': 27, 'format': 'PDF 1.4', 'title': 'OS', 'author': 'Pubg Rock', 'subject': '', 'keywords': 'DAG-P4AlC1g,BAE2iUPG4V0,0', 'moddate': '2026-01-23T18:41:08+00:00', 'trapped': '', 'modDate': "D:20260123184108+00'00'", 'creationDate': "D:20260123184110+00'00'", 'page': 0, 'source_file': 'OS 10_5Marks with ans.pdf', 'file_type': 'pdf'}, page_content='Operating System\n1.Elaborate on operating system services?\n Provide a set of services that act as an interface between users/applications and the \ncomputer hardware\nThese services make systems convenient to use, efficient, reliable, and secure.\n1. User Interface (UI) - Interactions between the user and computer\n2. Program Execution - Running application\n3. Process Management - The OS manages multiple 

In [ ]:
### split the documents into smaller chunks
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"split {len(documents)} douments into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"content:{split_docs[0].page_content[:200]}...")
        print(f"Metadata:{split_docs[0].metadata}...")

    return split_docs

chunks = split_documents(pdf_documents_list)
chunks

split 29 douments into 39 chunks

Example chunk:
content:Operating System
1.Elaborate on operating system services?
 Provide a set of services that act as an interface between users/applications and the 
computer hardware
These services make systems conveni...
Metadata:{'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-23T18:41:10+00:00', 'source': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf', 'file_path': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf', 'total_pages': 27, 'format': 'PDF 1.4', 'title': 'OS', 'author': 'Pubg Rock', 'subject': '', 'keywords': 'DAG-P4AlC1g,BAE2iUPG4V0,0', 'moddate': '2026-01-23T18:41:08+00:00', 'trapped': '', 'modDate': "D:20260123184108+00'00'", 'creationDate': "D:20260123184110+00'00'", 'page': 0, 'source_file': 'OS 10_5Marks with ans.pdf', 'file_type': 'pdf'}...


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-01-23T18:41:10+00:00', 'source': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf', 'file_path': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf', 'total_pages': 27, 'format': 'PDF 1.4', 'title': 'OS', 'author': 'Pubg Rock', 'subject': '', 'keywords': 'DAG-P4AlC1g,BAE2iUPG4V0,0', 'moddate': '2026-01-23T18:41:08+00:00', 'trapped': '', 'modDate': "D:20260123184108+00'00'", 'creationDate': "D:20260123184110+00'00'", 'page': 0, 'source_file': 'OS 10_5Marks with ans.pdf', 'file_type': 'pdf'}, page_content='Operating System\n1.Elaborate on operating system services?\n Provide a set of services that act as an interface between users/applications and the \ncomputer hardware\nThese services make systems convenient to use, efficient, reliable, and secure.\n1. User Interface (UI) - Interactions between the user and computer\n2. Program Execution - Running application\n3. Process Management - The OS manages multiple 

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializing the embedding manager
        Args: 

            model name: Hugging face name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence Transformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"model loaded sucessfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self,texts: List[str]) -> np.array:
        """Generate embeddings for a list of texts
            Args:
                texts: List of text string to embed
            returns:
                numpy array of embeddings with shape (len(texts)), embedding_dim")
        """
        if not self.model:
            raise ValueError("model not loaded")

        print(f"Generate embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

### Initialize the embedding manger
Embedding_Manager = EmbeddingManager()
Embedding_Manager


Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1860.64it/s]


model loaded sucessfully. Embedding dimension: 384


In [7]:
###Vectorstore

In [ ]:
class vectorestore:
    """Manages document embedding and stores into the vectore store"""

    def __init__(self, collection_name: str="pdf_documents", persist_directory: str = "../data/vectore_store"):
        """
        INitialize the vector store

        collection name: Name of the chromaDB collection
        persist_directory: Directory to persist the vectore store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize Chroma db client and collection"""

        try:
            #create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True) 
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            #set or create a collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "pdf documents embeddings for RAG"}
            )
            print(f"vector store initialized. collection:{self.collection_name}")
            print(f"Existing documents in collection:{self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self,documents: List[Any], embeddings: np.ndarray):
        """Add Documents and embedding to the vector store
        
        Args:
            documents: List of langchain documents
            embedding: corresponding embedding to the documeents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids =[]
        metadatas =[]
        documents_text = []
        embeddings_list =[]

        for i, (doc, embeddings) in enumerate(zip(documents, embeddings)):

            #Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #DOCUMENT content

            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embeddings.tolist())

            #Add to collection
            try:
                self.collection.add(
                    ids = ids,
                    embeddings= embeddings_list,
                    metadatas=metadatas,
                    documents= documents_text

                )

                print(f"sucessfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")

            except Exception as e:
                print(f"Error in adding documents to the vector store:{e} ")
                raise

vectore_store = vectorestore()
vectore_store

vector store initialized. collection:pdf_documents
Existing documents in collection:663


In [9]:
texts =[doc.page_content for doc in chunks]
Embedding_Manager = EmbeddingManager()
embeddings = Embedding_Manager.generate_embeddings(texts)
vectore_store.add_documents(chunks, embeddings)


Loading embedding model:all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3900.52it/s]


model loaded sucessfully. Embedding dimension: 384
Generate embeddings for 39 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.14it/s]


Generated embedding with shape: (39, 384)
Adding 39 documents to vector store...
sucessfully added 39 documents to vector store
Total documents in collection: 664
sucessfully added 39 documents to vector store
Total documents in collection: 665
sucessfully added 39 documents to vector store
Total documents in collection: 666
sucessfully added 39 documents to vector store
Total documents in collection: 667
sucessfully added 39 documents to vector store
Total documents in collection: 668
sucessfully added 39 documents to vector store
Total documents in collection: 669
sucessfully added 39 documents to vector store
Total documents in collection: 670
sucessfully added 39 documents to vector store
Total documents in collection: 671
sucessfully added 39 documents to vector store
Total documents in collection: 672
sucessfully added 39 documents to vector store
Total documents in collection: 673
sucessfully added 39 documents to vector store
Total documents in collection: 674
sucessfully added

In [10]:
class RAGRetriever:
    """Handles query based retrieval from the vector store"""
    def __init__(self, vectore_store: vectorestore, embedding_manager: EmbeddingManager ):
        self.vectore_store = vectore_store
        self.embedding_manager = embedding_manager

    def retrive(self, query: str, top_k: int =5, score_threshold: float = 0.0)-> List[Dict[str, Any]]:
            """
            Retrieve relevant documents for a query

            Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

            Returns:
            List of dictionaries containing retrieved documents and metadata
            """

            print(f"Retrieving documents for query: '{query}'")
            print(f"Top K: {top_k}, Score threshold: {score_threshold}")
            #Generate query embedding
            query_embedding = self.embedding_manager.generate_embeddings([query])[0]
            print(type(query_embedding))
            print(query_embedding.shape)
            print(query_embedding.dtype)
            print(query_embedding is None)

            #Search in vector store
            try:
                results = self.vectore_store.collection.query(
                      query_embeddings = [query_embedding.tolist()],
                      n_results= top_k
                 )

                 #process results
                retrieved_docs=[]
                print(results['documents'])
                #print(results['document'][0])
                
                if results['documents'][0]:
                    documents = results['documents'][0]
                    metadatas = results['metadatas'][0]
                    distances = results['distances'][0]
                    ids = results['ids'][0]

                    for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                        #convert distances using similarity score (chromaDB uses cosine distance)
                        similarity_score = 1- distance

                        if similarity_score >= score_threshold:
                               retrieved_docs.append({
                               'id': doc_id,
                               'content': document,
                               'metadata': metadata,
                               'similarity_score': similarity_score,
                               'distance': distance,
                               'rank': i + 1
                               })
                    print(f"Reterived {len(retrieved_docs)} documents (after filtering)")
                else:
                     print("No documents found")
                return retrieved_docs
                
            except Exception as e:
                 print(f"Error during Reterieval: {e}")
                 return []

In [11]:
RAG_Reteriever = RAGRetriever(vectore_store, Embedding_Manager)
RAG_Reteriever

In [12]:
RAG_Reteriever.retrive("Write short notes on time sharing system")

Retrieving documents for query: 'Write short notes on time sharing system'
Top K: 5, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.20it/s]

Generated embedding with shape: (1, 384)
<class 'numpy.ndarray'>
(384,)
float32
False
[['Level\nSpeed\nCost\nCapacity\nRegisters\nVery High\nVery High\nVery Low\nCache\nHigh\nHigh\nLow\nMain Memory\nMedium\nMedium\nMedium\nSecondary Storage\nLow\nLow\nHigh\nTertiary Storage\nVery Low\nVery Low\nVery High\n5 MARK\n1. Write short notes on time sharing system.\nA time sharing system is an operating system technique in which multiple users or \nprocesses share the CPU simultaneously by\nDividing CPU time into small units called time slices\nEach process gets the CPU for a short duration, creating the illusion that all processes \nare executing at the same time.\nFeatures of Time Sharing System\nMulti-user support\nRapid context switching\nPreemptive CPU scheduling\nInteractive response\nFair CPU allocation', 'Level\nSpeed\nCost\nCapacity\nRegisters\nVery High\nVery High\nVery Low\nCache\nHigh\nHigh\nLow\nMain Memory\nMedium\nMedium\nMedium\nSecondary Storage\nLow\nLow\nHigh\nTertiary Stora

[{'id': 'doc_5cf3a1dd_24',
  'content': 'Level\nSpeed\nCost\nCapacity\nRegisters\nVery High\nVery High\nVery Low\nCache\nHigh\nHigh\nLow\nMain Memory\nMedium\nMedium\nMedium\nSecondary Storage\nLow\nLow\nHigh\nTertiary Storage\nVery Low\nVery Low\nVery High\n5 MARK\n1. Write short notes on time sharing system.\nA time sharing system is an operating system technique in which multiple users or \nprocesses share the CPU simultaneously by\nDividing CPU time into small units called time slices\nEach process gets the CPU for a short duration, creating the illusion that all processes \nare executing at the same time.\nFeatures of Time Sharing System\nMulti-user support\nRapid context switching\nPreemptive CPU scheduling\nInteractive response\nFair CPU allocation',
  'metadata': {'creationDate': "D:20260123184110+00'00'",
   'source_file': 'OS 10_5Marks with ans.pdf',
   'file_path': '..\\data\\pdf_files\\OS 10_5Marks with ans.pdf',
   'doc_index': 24,
   'creationdate': '2026-01-23T18:41:10+0

#Integration VectorDB pipeline with LLM output

In [13]:
#simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = "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"
llm = ChatGroq(groq_api_key = groq_api_key, model_name = "openai/gpt-oss-20b", temperature = 0.1, max_tokens=1024)
def Rag_simple(query, retriever, llm, top_k =3):
    #reteriver the context
    results = retriever.retrive(query, top_k= top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevent context found to answer the question."
    ## Generate the answer uisng Groq LLM
    prompt = f"""use the following context to answer the question concisely
    
    context{context}

    Question:{query}

    Answer:"""

    response = llm.invoke([prompt])
    return response.content

In [14]:
answer = Rag_simple("Elaborate on operating system services?", RAG_Reteriever, llm)
print(answer)

Retrieving documents for query: 'Elaborate on operating system services?'
Top K: 3, Score threshold: 0.0
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.05it/s]

Generated embedding with shape: (1, 384)
<class 'numpy.ndarray'>
(384,)
float32
False
[['Operating System\n1.Elaborate on operating system services?\n Provide a set of services that act as an interface between users/applications and the \ncomputer hardware\nThese services make systems convenient to use, efficient, reliable, and secure.\n1. User Interface (UI) - Interactions between the user and computer\n2. Program Execution - Running application\n3. Process Management - The OS manages multiple processes efficiently\n4. Memory Management - The OS manages the system’s main memory (RAM).\n5. File System Management - The OS organizes and manages data stored on disks. (Creating, \ndeleting, reading, and writing files)\n6. I/O Device Management - Controls input and output devices through device drivers.\n7. Communication Services - Enables communication between processes. (Inter-process \ncommunication, Network communication:)\n8. Error Detection and Handling - The OS constantly monitors th

**Operating‑System Services**

| Service | What it does | Why it matters |
|--------|--------------|----------------|
| **User Interface (UI)** | Provides the graphical or command‑line interface through which users interact with the system. | Makes the computer usable and accessible. |
| **Program Execution** | Loads, starts, and terminates applications, managing their life cycles. | Enables multitasking and efficient use of CPU time. |
| **Process Management** | Creates, schedules, synchronizes, and terminates processes; handles inter‑process communication. | Keeps multiple programs running smoothly and fairly. |
| **Memory Management** | Allocates, protects, and de‑allocates RAM for processes; handles paging/segmentation. | Maximizes memory utilization and prevents conflicts. |
| **File System Management** | Organizes, creates, deletes, reads, and writes files on storage devices. | Provides persistent data storage and a logical structure. |
| **I/O Device Management** | Controls inpu

In [15]:
from groq import Groq

groq_api_key = "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"
client = Groq(api_key=groq_api_key)

models = client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
qwen/qwen3.8-27b
canopylabs/orpheus-arabic-saudi
groq/compound
openai/gpt-oss-safeguard-20b
qwen/qwen3.6-27b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
groq/compound-mini
openai/gpt-oss-20b
canopylabs/orpheus-v1-english
openai/gpt-oss-120b


In [16]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context= False):
    """
    RAG Pipeline with extra features:
    - Returns answers, sources, confidence score, and optionally fully context.
    """
    results = retriever.retrive(query, top_k= top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources' : [], 'confidence': 0.0, 'context': ''}

    #Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [
    {
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results
    ]

    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"""use the following context to answer the question concisely
        
        context\n{context}
    
        Question:\n{query}
    
        Answer:"""
    response = llm.invoke([prompt])
    output ={
        'answer': response.content,
        'sources':sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context

    return output

result = rag_advanced("Elaborate on operating system services?", RAG_Reteriever, llm, top_k=3, min_score =0.1, return_context = True)
print(result)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context' ] [: 300])

Retrieving documents for query: 'Elaborate on operating system services?'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.18it/s]

Generated embedding with shape: (1, 384)
<class 'numpy.ndarray'>
(384,)
float32
False
[['Operating System\n1.Elaborate on operating system services?\n Provide a set of services that act as an interface between users/applications and the \ncomputer hardware\nThese services make systems convenient to use, efficient, reliable, and secure.\n1. User Interface (UI) - Interactions between the user and computer\n2. Program Execution - Running application\n3. Process Management - The OS manages multiple processes efficiently\n4. Memory Management - The OS manages the system’s main memory (RAM).\n5. File System Management - The OS organizes and manages data stored on disks. (Creating, \ndeleting, reading, and writing files)\n6. I/O Device Management - Controls input and output devices through device drivers.\n7. Communication Services - Enables communication between processes. (Inter-process \ncommunication, Network communication:)\n8. Error Detection and Handling - The OS constantly monitors th

{'answer': '**Operating‑System Services – What They Do**\n\n| Service | What it Provides | Why It Matters |\n|---------|------------------|----------------|\n| **User Interface (UI)** | Graphical or command‑line interface for users to interact with the system. | Makes the computer usable and accessible. |\n| **Program Execution** | Loads, starts, and terminates applications; schedules CPU time. | Enables multitasking and efficient use of processor resources. |\n| **Process Management** | Creates, synchronizes, and terminates processes; handles context switching. | Keeps multiple programs running without interference. |\n| **Memory Management** | Allocates, protects, and de‑allocates RAM; handles paging/virtual memory. | Maximizes available memory and prevents crashes. |\n| **File System Management** | Organizes, creates, deletes, reads, and writes files on storage devices. | Provides persistent data storage and file organization. |\n| **I/O Device Management** | Controls input/output d

In [21]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.RAG_Reteriever = retriever
        self.llm= llm
        self.history =[] #store query history
    def query(self, question:str, top_k: int=5, min_score: float =0.2, stream:bool = False, summarize: bool = False) -> List[Dict[str, Any]]:
        #Reterive Relevant Documents
        results = self.RAG_Reteriever.retrive(question, top_k = top_k, score_threshold = min_score)
        if not results:
            answer = "No relevant answer found"
            sources =[]
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [
            {
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results
            ]
        #Streaming Answer simulation
        prompt = f"""use the following context to answer the question concisely
                
                context:\n{context}
            
                Question:\n{question}
            
                Answer:"""
        if stream:
            print("Streaming Answer:")
            for i in range(0, len(prompt), 80):
                print(prompt[i: i+80], end =' ', flush=True)
                time.sleep(0.05)
            print()
            response = self.llm.invoke([prompt])
            answer = response.content

        #Add Citation to the answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 Sentences: \n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content
        self.history.append({
            'question': question, 
            'answer': answer,
            'sources': sources, 
            'summary': summary
        })

        return {
            'question': question, 
            'answer': answer,
            'sources': sources, 
            'summary': summary,
            'history': self.history
        }
    #Example Usage:
adv_rag = AdvancedRAGPipeline(RAG_Reteriever, llm)
result = adv_rag.query("Elaborate on operating system services?", top_k =3, min_score =0.1, stream = True, summarize = True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Elaborate on operating system services?'
Top K: 3, Score threshold: 0.1
Generate embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 35.34it/s]

Generated embedding with shape: (1, 384)
<class 'numpy.ndarray'>
(384,)
float32
False
[['Operating System\n1.Elaborate on operating system services?\n Provide a set of services that act as an interface between users/applications and the \ncomputer hardware\nThese services make systems convenient to use, efficient, reliable, and secure.\n1. User Interface (UI) - Interactions between the user and computer\n2. Program Execution - Running application\n3. Process Management - The OS manages multiple processes efficiently\n4. Memory Management - The OS manages the system’s main memory (RAM).\n5. File System Management - The OS organizes and manages data stored on disks. (Creating, \ndeleting, reading, and writing files)\n6. I/O Device Management - Controls input and output devices through device drivers.\n7. Communication Services - Enables communication between processes. (Inter-process \ncommunication, Network communication:)\n8. Error Detection and Handling - The OS constantly monitors th

 secure.
1. User Interface (UI) - Interactions between the user and computer
2.  Program Execution - Running application
3. Process Management - The OS manages m ultiple processes efficiently
4. Memory Management - The OS manages the system’s  main memory (RAM).
5. File System Management - The OS organizes and manages dat a stored on disks. (Creating, 
deleting, reading, and writing files)
6. I/O Devi ce Management - Controls input and output devices through device drivers.
7. Com munication Services - Enables communication between processes. (Inter-process 
c ommunication, Network communication:)
8. Error Detection and Handling - The OS c onstantly monitors the system for errors.
(Hardware failures, I/O errors)

Opera ting System
1.Elaborate on operating system services?
 Provide a set of services  that act as an interface between users/applications and the 
computer hardware
 These services make systems convenient to use, efficient, reliable, and secure.
 1. User Interface (UI) - Int